# 00 Load Library
---


In [ ]:
from pathlib import Path
import pandas as pd


# 01 Inisialisasi Path dan Konsolidasi Seluruh Artefak
---
## Konfigurasi Engine Inferensi
---
### Pembacaan Hasil Komputasi Pilar 1, 2, dan 3
---
Membaca tabel skor kelayakan, tipologi klaster wilayah, dan sinyal arah tren untuk inferensi keputusan real-time.


In [ ]:
BASE_DIR = Path.cwd().parent.parent if Path.cwd().name == 'analisis' else Path.cwd()
CLEANED_DIR = BASE_DIR / 'data' / 'cleaned'
HASIL_DIR = BASE_DIR / 'outputs' / 'hasil'
df_skor = pd.read_csv(HASIL_DIR / 'hasil_skoring_sektor.csv')
df_cluster = pd.read_csv(HASIL_DIR / 'hasil_clustering_wilayah.csv')
df_forecast = pd.read_csv(HASIL_DIR / 'hasil_forecasting_sektor.csv')
print('Seluruh artefak analitik 3 pilar berhasil dimuat.')


# 02 Implementasi Fungsi Inferensi dan Validasi Luas Ruko
---
## Logika Pencocokan Aturan
---
### Perhitungan Budget Feasibility Ratio BFR dan Standar Luas
---
Menghitung rasio kecukupan anggaran terhadap standar minimum operasional ruko UMKM 30 meter persegi.


In [ ]:
def rekomendasi_umkm(sektor, modal_sewa=None, wilayah=None):
    df_s = df_skor[df_skor['sektor'] == sektor].copy()
    df_m = df_s.merge(df_cluster[['wilayah', 'karakteristik_pasar', 'tarif_sewa_per_m2_tahun']], on='wilayah')
    tren_row = df_forecast[(df_forecast['sektor'] == sektor) & (df_forecast['tahun'] == 2026)].iloc[0]
    if wilayah:
        w_data = df_m[df_m['wilayah'] == wilayah].iloc[0]
        tarif = w_data['tarif_sewa_per_m2_tahun_x']
        luas_tercover = (modal_sewa / tarif) if modal_sewa else 0
        bfr = (modal_sewa / (tarif * 30)) if modal_sewa else 1.0
        status_budget = 'MEMADAI' if bfr >= 1.0 else ('MARGINAL' if bfr >= 0.7 else 'KURANG MEMADAI')
        return {
            'mode': 'wilayah_spesifik',
            'wilayah': wilayah,
            'sektor': sektor,
            'skor': w_data['skor_kelayakan'],
            'kategori': w_data['kategori'],
            'klaster': w_data['karakteristik_pasar'],
            'tren_sektor': tren_row['status_arah_tren'],
            'estimasi_luas_m2': round(luas_tercover, 1),
            'status_budget': status_budget
        }
    else:
        df_m = df_m.sort_values(by='skor_kelayakan', ascending=False)
        top_3 = df_m.head(3).to_dict('records')
        return {
            'mode': 'wilayah_terbuka',
            'sektor': sektor,
            'tren_sektor': tren_row['status_arah_tren'],
            'top_3_rekomendasi': top_3
        }


# 03 Pengujian Skenario 1 Target Wilayah Spesifik
---
## Simulasi Input Calon Wirausahawan
---
### Kafe di Kabupaten Sleman dengan Modal Tiga Puluh Juta
---
Menguji respons inferensi untuk preferensi wilayah tertentu disertai verifikasi kelayakan anggaran sewa.


In [ ]:
hasil_skenario_1 = rekomendasi_umkm(sektor='cafe', modal_sewa=30000000, wilayah='sleman')
for k, v in hasil_skenario_1.items():
    print(f'{k}: {v}')


# 04 Pengujian Skenario 2 Wilayah Terbuka Top 3
---
## Simulasi Pencarian Lokasi Terbaik
---
### Rekomendasi Sektor Laundry di Seluruh D.I. Yogyakarta
---
Menguji respons inferensi pemeringkatan multi-wilayah untuk wirausahawan yang belum menentukan lokasi spesifik.


In [ ]:
hasil_skenario_2 = rekomendasi_umkm(sektor='laundry', modal_sewa=None, wilayah=None)
print('Sektor:', hasil_skenario_2['sektor'], '| Tren:', hasil_skenario_2['tren_sektor'])
for rank, rec in enumerate(hasil_skenario_2['top_3_rekomendasi'], 1):
    print(f'Peringkat {rank}: {rec["wilayah"]} | Skor: {rec["skor_kelayakan"]} ({rec["kategori"]}) | {rec["karakteristik_pasar"]}')
